# Showing a distribution

**Showing a distribution -- histogram, box, violin, and the honest strip plot.**

> **The question this chart answers:** "what is the shape of this data?"

**What it shows:**

- bin width changes the story a histogram tells -- see it happen
- a box plot hides shape; two very different datasets can share one
- a violin shows shape but invents smoothness it cannot know
- with few points, just show the points

The rule underneath: a summary throws information away. Know what you threw.

---

*Chapter:* `choosing` — which chart answers which question  
*Run the cells in order.* Every figure is also written to `viz/output/choosing/`, which is what the Streamlit gallery (`viz/project/gallery.py`) reads.


## Setup

These lines are how every notebook in the folder finds `vizkit.py`, which holds the save helpers and the seeded sample data. The data is seeded on purpose: your figures should come out identical to everyone else's.

`save()` writes each figure into `viz/output/` **and** leaves it on screen here. The trailing `;` on those calls only stops the notebook echoing the path it returns.


In [ ]:
%matplotlib inline

# A notebook has no __file__, so find viz/ by walking up from this
# notebook's own folder until vizkit.py turns up.
import sys
from pathlib import Path

VIZ = next(p for p in [Path.cwd(), *Path.cwd().parents]
           if (p / "vizkit.py").exists())
sys.path.insert(0, str(VIZ))

import matplotlib.pyplot as plt
import numpy as np

from vizkit import save, temperatures

# Where save() files this lesson's output: viz/output/choosing/
LESSON = "choosing/distribution"


## The data

365 daily temperatures, as a plain NumPy array. One column of numbers is all a distribution chart ever needs.


In [ ]:
temps = temperatures()["temp_c"].to_numpy()


## 1. The bin width IS an argument you are making

Watch the shape change as the bin count goes 3 → 10 → 30 → 200. None of these panels is wrong, and none of them is *the* answer: the bin width is a choice you are making on the reader's behalf.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2), sharey=False)

for ax, bins in zip(axes, [3, 10, 30, 200]):
    ax.hist(temps, bins=bins, color="#4C72B0", edgecolor="white")
    ax.set_title(f"bins = {bins}")
    ax.set_xlabel("temp (C)")

fig.suptitle("Same 365 numbers. Too few bins hides the shape; too many shows noise.",
             fontsize=11)
fig.tight_layout()
save(fig, LESSON, "bin-width");


## 2. The box plot's blind spot

The three datasets below were built on purpose to share a five-number summary. If the box plot could see shape, the top panel would look like three different things. It does not.


In [ ]:
# Three datasets, deliberately built to share a five-number summary while
# looking nothing alike.
# The parameters are chosen so all three share a five-number summary: their
# quartiles agree to within about 2 points. That is the whole point -- the
# box plot cannot tell them apart, and the histogram below cannot miss.
rng = np.random.default_rng(7)
normal = rng.normal(50, 11.86, 400)                       # IQR ~16
bimodal = np.concatenate([rng.normal(42, 3.5, 200),       # two tight humps
                          rng.normal(58, 3.5, 200)])      # sitting on Q1 and Q3
uniform = rng.uniform(34, 66, 400)                        # flat, same IQR
groups = [normal, bimodal, uniform]
labels = ["one hump", "two humps", "flat"]

# One wide panel for the boxes, then one panel per histogram. Overlaying the
# three histograms turns them to mud -- separate panels is the whole trick,
# and it is what the layout chapter calls small multiples.
fig, axes = plt.subplot_mosaic([["box", "box", "box"],
                                ["h0", "h1", "h2"]], figsize=(11, 6))

axes["box"].boxplot(groups, tick_labels=labels, vert=False)
axes["box"].set_title("Box plots: near-identical quartiles")
axes["box"].set_xlim(5, 85)

for i, (values, label) in enumerate(zip(groups, labels)):
    ax = axes[f"h{i}"]
    ax.hist(values, bins=35, color="#4C72B0", edgecolor="white")
    ax.set_title(label)
    ax.set_xlim(5, 85)              # same axis, or the comparison is a lie
    ax.set_ylim(0, 45)
    if i:
        ax.set_yticklabels([])

fig.suptitle("Same five-number summary. Three completely different shapes.",
             fontsize=12)
axes["h0"].set_ylabel("count")
fig.tight_layout()
save(fig, LESSON, "box-hides-shape");


## 3. With few points, show the points

With fourteen points, a box plot draws quartiles from three or four observations each, and a violin draws a smooth curve through gaps where there is no data at all. Show the points instead.


In [ ]:
small = rng.normal(50, 12, 14)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))

axes[0].boxplot([small], tick_labels=["n=14"])
axes[0].set_title("Box: implies more data than exists")

axes[1].violinplot([small])
axes[1].set_title("Violin: invents a smooth curve")

axes[2].plot(np.ones(len(small)) + rng.normal(0, 0.02, len(small)), small,
             "o", alpha=0.7, color="#4C72B0")
axes[2].set_xlim(0.8, 1.2)
axes[2].set_xticks([])
axes[2].set_title("Strip: 14 points, shown as 14 points")

fig.tight_layout()
save(fig, LESSON, "few-points");


## Rules of thumb

```text
Choose by sample size and question:
  n < ~30            -> show every point (strip / dot)
  "what shape?"      -> histogram, and try more than one bin width
  "compare groups?"  -> box or violin, but say n somewhere
```


## Try it yourself

Edit the cells above and re-run them — that is what the notebook is for.

1. In section 1, add a fifth panel with `bins="auto"` (matplotlib will pick with the Freedman–Diaconis rule). Which of the hand-picked bin counts does it land nearest?
2. In section 2, change `bimodal`'s two humps to be 4 points apart instead of 16. At what separation does the box plot start to notice?
3. In section 3, raise `n` from 14 to 400 in the `small` sample. At roughly what n does the strip plot become the unreadable one?


In [ ]:
# your turn


---

**Previous:** [`choosing/comparison`](comparison.ipynb)  
**Next:** [`choosing/relationship`](relationship.ipynb)
